# LLM Feature Extraction — Schema v2 (Claude Haiku)

Runs the full structured feature extraction over every scraped article body, using the Anthropic Batches API (50% discount) with prompt caching on the shared system prompt and tool schema. Each article is forced through the `extract_article_features` tool (Schema v2: the `usable` flag, composite `sentiment_score`, the three orthogonal channels `supply_impact` / `demand_impact` / `risk_premium`, plus `magnitude`, `certainty`, `event_type`, `time_horizon`, `entities`), and the parsed result is written to the `llm_features` table.

The prompt and tool schema are imported locked from `03_src/nlp/llm_features.py` (validated during calibration), never redefined here. The pipeline is resumable at two levels: it skips `article_id`s already in `llm_features`, and it persists the `batch_id` to a file so re-running polls the existing batch instead of resubmitting. There is no `body_valid` filter; the LLM's own `usable` flag is the canonical downstream filter.

## Cell 1 — Setup

Loads credentials and the Anthropic client, and defines the canonical paths: the DB, the persisted `batch_id` file (the resume checkpoint), the error log, and the logbook.

In [ ]:
import sys, os, json, sqlite3, time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
import anthropic

load_dotenv(Path('../.env').resolve())
sys.path.insert(0, str(Path('../03_src').resolve()))

client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])

DB_PATH          = Path('../01_data/wti_thesis.db')
BATCH_ID_FILE    = Path('../05_reports/calibration/llm_batch_id.txt')
ERROR_LOG_FILE   = Path('../05_reports/calibration/batch_errors.json')
LOGBOOK_PATH     = Path('../05_reports/development-decisions/project_logbook.md')

print('Client ready.')

## Cell 2 — Configuration (imports from shared module)

Imports the locked `SYSTEM_PROMPT` and `EXTRACTION_TOOL` from `nlp.llm_features` rather than redefining them, and sets the run constants: `claude-haiku-4-5`, `BODY_LIMIT=1500` chars (matching calibration), and `COMMIT_EVERY=500`.

In [16]:
# Import the locked constants from the validated module — do not redefine here
from nlp.llm_features import SYSTEM_PROMPT, EXTRACTION_TOOL

MODEL          = 'claude-haiku-4-5'
MAX_TOKENS     = 500
BODY_LIMIT     = 1500       # chars — matches calibration
COMMIT_EVERY   = 500        # rows per DB transaction
POLL_INTERVAL  = 60         # seconds between status checks
MAX_POLL_HOURS = 24         # timeout safety

print(f'Model:          {MODEL}')
print(f'Max tokens:     {MAX_TOKENS}')
print(f'Body truncation:{BODY_LIMIT} chars')
print(f'System prompt:  {len(SYSTEM_PROMPT):,} chars')
print(f'Tool schema:    {len(json.dumps(EXTRACTION_TOOL)):,} chars')

Model:          claude-haiku-4-5
Max tokens:     500
Body truncation:1500 chars
System prompt:  3,445 chars
Tool schema:    2,869 chars


## Cell 3 — Query articles to process

Selects articles with a real body (`body IS NOT NULL AND body NOT LIKE 'ERROR%'`) and subtracts those already in `llm_features`, so `df_todo` holds only what this run needs to submit. Note the deliberate absence of a `body_valid` filter: the LLM `usable` flag is the canonical one.

In [29]:
SQL_FILTER = """
    SELECT id, title, body
    FROM articles
    WHERE body IS NOT NULL
      AND body NOT LIKE 'ERROR%'
"""
# ↑ No body_valid filter — usable flag in llm_features is the canonical filter

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql(SQL_FILTER, conn)

already_done = set(
    r[0] for r in conn.execute(
        'SELECT article_id FROM llm_features WHERE article_id IS NOT NULL'
    ).fetchall()
)
conn.close()

df_todo = df[~df['id'].isin(already_done)].reset_index(drop=True)

print(f'SQL filter: body IS NOT NULL AND body NOT LIKE \'ERROR%\'  (no body_valid filter)')
print(f'Total matching articles: {len(df):,}')
print(f'Already in llm_features: {len(already_done):,}')
print(f'To submit this run:      {len(df_todo):,}')
print()
# Date range requires joining to articles.datetime — show via a quick query
conn = sqlite3.connect(DB_PATH)
rng = pd.read_sql(
    """SELECT MIN(a.datetime) as min_dt, MAX(a.datetime) as max_dt
       FROM articles a
       WHERE a.body IS NOT NULL AND a.body NOT LIKE 'ERROR%'""",
    conn
)
conn.close()
print(f'Date range: {rng.iloc[0]["min_dt"]} → {rng.iloc[0]["max_dt"]}')

SQL filter: body IS NOT NULL AND body NOT LIKE 'ERROR%'  (no body_valid filter)
Total matching articles: 19,619
Already in llm_features: 19,512
To submit this run:      107

Date range: 2024-01-01 00:30:00+00:00 → 2026-05-12 00:30:00+00:00


## Cell 4 — Build batch requests

Builds one request per article, forcing the `extract_article_features` tool (`tool_choice`) and marking the system prompt cacheable (`cache_control: ephemeral`) so the shared prompt and schema are billed once and read cheaply for every article. `batch_requests` is the payload list; the cell prints the first request's structure for verification.

In [30]:
def make_user_text(title: str, body) -> str:
    t = str(title).encode('utf-8', errors='ignore').decode('utf-8').strip()
    b = ''
    if isinstance(body, str):
        b = body.encode('utf-8', errors='ignore').decode('utf-8')[:BODY_LIMIT].strip()
    return f'Title: {t}\n\nBody: {b}' if b else f'Title: {t}'

def make_request(article_id: int, title: str, body) -> dict:
    return {
        'custom_id': str(article_id),
        'params': {
            'model': MODEL,
            'max_tokens': MAX_TOKENS,
            'system': [
                {
                    'type': 'text',
                    'text': SYSTEM_PROMPT,
                    'cache_control': {'type': 'ephemeral'},
                }
            ],
            'tools': [EXTRACTION_TOOL],
            'tool_choice': {'type': 'tool', 'name': 'extract_article_features'},
            'messages': [
                {'role': 'user', 'content': make_user_text(title, body)}
            ],
        },
    }

batch_requests = [
    make_request(int(row['id']), row['title'], row['body'])
    for _, row in df_todo.iterrows()
]

# Show payload structure for first article (for format verification)
first = batch_requests[0]
preview = {
    'custom_id': first['custom_id'],
    'params': {
        'model': first['params']['model'],
        'max_tokens': first['params']['max_tokens'],
        'system': [
            {
                'type': first['params']['system'][0]['type'],
                'text': first['params']['system'][0]['text'][:120] + '...',
                'cache_control': first['params']['system'][0]['cache_control'],
            }
        ],
        'tools': [{'name': first['params']['tools'][0]['name'], '...': '(full schema omitted)'}],
        'tool_choice': first['params']['tool_choice'],
        'messages': [
            {
                'role': 'user',
                'content': first['params']['messages'][0]['content'][:200] + '...',
            }
        ],
    },
}
print(f'Total requests built: {len(batch_requests):,}')
print()
print('=== First request payload structure ===')
print(json.dumps(preview, indent=2))

Total requests built: 107

=== First request payload structure ===
{
  "custom_id": "9549",
  "params": {
    "model": "claude-haiku-4-5",
    "max_tokens": 500,
    "system": [
      {
        "type": "text",
        "text": "You are a financial analyst specializing in WTI crude oil markets. Your task is to analyze news articles and extract str...",
        "cache_control": {
          "type": "ephemeral"
        }
      }
    ],
    "tools": [
      {
        "name": "extract_article_features",
        "...": "(full schema omitted)"
      }
    ],
    "tool_choice": {
      "type": "tool",
      "name": "extract_article_features"
    },
    "messages": [
      {
        "role": "user",
        "content": "Title: US Oil Refiners Brace For A Tough Year As Investor Sentiment Turns Negative\n\nBody: US refiner profits started to fall toward the end of 2023 as new refining capacity came online and margins ret..."
      }
    ]
  }
}


## Cell 5 — Cost estimate (review before submitting)

Estimates the batch cost from token counts and Batches-API pricing (one cache write, per-article cache reads, non-cached input, output), assuming a roughly 60% usable share. A review gate before the actual submission.

In [31]:
# Anthropic Haiku 4.5 pricing (Batches API = 50% off standard input + output)
# Standard:        input $0.80/MTok | output $4.00/MTok
#                  cache_write $1.00/MTok | cache_read $0.08/MTok
# Batches (50%):   input $0.40/MTok | output $2.00/MTok
#                  cache_write $0.50/MTok | cache_read $0.04/MTok

PRICE_INPUT_BATCH       = 0.40 / 1e6   # $/token
PRICE_OUTPUT_BATCH      = 2.00 / 1e6
PRICE_CACHE_WRITE       = 0.50 / 1e6
PRICE_CACHE_READ_BATCH  = 0.04 / 1e6

N = len(batch_requests)

# Cached block: system prompt + tool schema (written once, read N times)
cached_block_tokens = (len(SYSTEM_PROMPT) + len(json.dumps(EXTRACTION_TOOL))) // 4
cache_write_cost  = cached_block_tokens * PRICE_CACHE_WRITE
cache_read_cost   = cached_block_tokens * N * PRICE_CACHE_READ_BATCH

# Per-article non-cached input (title + body truncated to BODY_LIMIT chars)
avg_article_tokens = int(
    df_todo.apply(
        lambda r: (len(str(r['title'])) + min(len(str(r['body'])), BODY_LIMIT)) / 4,
        axis=1
    ).mean()
)
input_cost = avg_article_tokens * N * PRICE_INPUT_BATCH

# Output: assume 60% usable (≈200 tok) and 40% not usable (≈12 tok)
est_usable_frac = 0.60
avg_output_tokens = int(est_usable_frac * 200 + (1 - est_usable_frac) * 12)
output_cost = avg_output_tokens * N * PRICE_OUTPUT_BATCH

total_est = cache_write_cost + cache_read_cost + input_cost + output_cost

print(f'Articles to submit:       {N:,}')
print(f'Cached block (tokens):    {cached_block_tokens:,}')
print(f'Avg article input tokens: {avg_article_tokens}')
print(f'Avg output tokens:        {avg_output_tokens}  (assumes {est_usable_frac:.0%} usable)')
print()
print(f'{"Item":<35} {"Cost":>10}')
print('-' * 47)
print(f'{"Cache write (1×)":<35} ${cache_write_cost:>9.4f}')
print(f'{"Cache reads ({N:,}×)":<35} ${cache_read_cost:>9.4f}')
print(f'{"Non-cached input":<35} ${input_cost:>9.4f}')
print(f'{"Output":<35} ${output_cost:>9.4f}')
print('-' * 47)
print(f'{"ESTIMATED TOTAL":<35} ${total_est:>9.4f}')
print()
print('⚠️  Review the payload structure (Cell 4) and this estimate.')
print('   Run Cell 6 to submit the batch.')

Articles to submit:       107
Cached block (tokens):    1,578
Avg article input tokens: 199
Avg output tokens:        124  (assumes 60% usable)

Item                                      Cost
-----------------------------------------------
Cache write (1×)                    $   0.0008
Cache reads ({N:,}×)                $   0.0068
Non-cached input                    $   0.0085
Output                              $   0.0265
-----------------------------------------------
ESTIMATED TOTAL                     $   0.0426

⚠️  Review the payload structure (Cell 4) and this estimate.
   Run Cell 6 to submit the batch.


## Cell 6 — Submit batch

Submits the batch and persists its `batch_id` to `llm_batch_id.txt`. If that file already exists the cell does not resubmit; it only reports the existing id. This is the top-level resume guard.

In [32]:
if BATCH_ID_FILE.exists():
    batch_id = BATCH_ID_FILE.read_text().strip()
    print(f'Batch already submitted: {batch_id}')
    print('Skip to Cell 7 (polling).')
else:
    submit_time = datetime.now(timezone.utc)
    batch = client.messages.batches.create(requests=batch_requests)
    batch_id = batch.id

    BATCH_ID_FILE.parent.mkdir(parents=True, exist_ok=True)
    BATCH_ID_FILE.write_text(batch_id)

    print(f'Batch submitted at: {submit_time.isoformat()}')
    print(f'Batch ID:           {batch_id}')
    print(f'Requests:           {batch.request_counts}')
    print(f'ID saved to:        {BATCH_ID_FILE}')

Batch submitted at: 2026-05-18T11:52:21.136696+00:00
Batch ID:           msgbatch_018JvAUYvymcoVVakpqA3f1T
Requests:           MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=107, succeeded=0)
ID saved to:        ../05_reports/calibration/llm_batch_id.txt


## Cell 7 — Poll until complete

Polls the batch every `POLL_INTERVAL` seconds, printing succeeded / errored / processing counts, until it reports `ended` or the `MAX_POLL_HOURS` timeout is reached.

In [33]:
batch_id = BATCH_ID_FILE.read_text().strip()
print(f'Polling batch: {batch_id}')

poll_start = time.time()
max_seconds = MAX_POLL_HOURS * 3600

while True:
    elapsed = time.time() - poll_start
    if elapsed > max_seconds:
        print(f'ERROR: batch did not complete within {MAX_POLL_HOURS}h. '
              f'batch_id={batch_id}')
        break

    batch = client.messages.batches.retrieve(batch_id)
    counts = batch.request_counts
    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(
        f'[{now}] status={batch.processing_status:12s} | '
        f'processing={counts.processing} succeeded={counts.succeeded} '
        f'errored={counts.errored} canceled={counts.canceled} expired={counts.expired}'
    )

    if batch.processing_status == 'ended':
        print(f'\nBatch complete. Elapsed: {elapsed/60:.1f} min')
        break

    time.sleep(POLL_INTERVAL)

Polling batch: msgbatch_018JvAUYvymcoVVakpqA3f1T
[2026-05-18 13:52:34] status=in_progress  | processing=107 succeeded=0 errored=0 canceled=0 expired=0
[2026-05-18 13:53:34] status=in_progress  | processing=107 succeeded=0 errored=0 canceled=0 expired=0
[2026-05-18 13:54:35] status=in_progress  | processing=107 succeeded=0 errored=0 canceled=0 expired=0
[2026-05-18 13:55:35] status=in_progress  | processing=107 succeeded=0 errored=0 canceled=0 expired=0
[2026-05-18 13:56:35] status=in_progress  | processing=107 succeeded=0 errored=0 canceled=0 expired=0
[2026-05-18 13:57:35] status=in_progress  | processing=107 succeeded=0 errored=0 canceled=0 expired=0
[2026-05-18 13:58:36] status=in_progress  | processing=107 succeeded=0 errored=0 canceled=0 expired=0
[2026-05-18 13:59:36] status=in_progress  | processing=107 succeeded=0 errored=0 canceled=0 expired=0
[2026-05-18 14:00:36] status=in_progress  | processing=107 succeeded=0 errored=0 canceled=0 expired=0
[2026-05-18 14:01:36] status=in_p

## Cell 8 — Retrieve and parse results

Streams the batch results and turns each into a DB-ready `row`. Non-succeeded results and missing `tool_use` blocks are collected into `errors` (written to `batch_errors.json`). For `usable=0` articles every channel field is set null (the schema short-circuits them); usable articles carry the full channel set, with `event_type` and `entities` stored as JSON strings.

In [34]:
batch_id = BATCH_ID_FILE.read_text().strip()

rows = []       # dicts ready for DB insertion
errors = []     # non-succeeded results

for result in client.messages.batches.results(batch_id):
    article_id = int(result.custom_id)

    if result.result.type != 'succeeded':
        errors.append({
            'article_id': article_id,
            'error_type': result.result.type,
            'error': str(getattr(result.result, 'error', '')),
        })
        continue

    tool_input = None
    for block in result.result.message.content:
        if block.type == 'tool_use':
            tool_input = block.input
            break

    if tool_input is None:
        errors.append({
            'article_id': article_id,
            'error_type': 'no_tool_use_block',
            'error': 'tool_use block missing from succeeded response',
        })
        continue

    usable = bool(tool_input.get('usable', False))
    row = {'article_id': article_id, 'usable': int(usable)}

    if usable:
        row['sentiment_score'] = tool_input.get('sentiment_score')
        row['supply_impact']   = tool_input.get('supply_impact')
        row['demand_impact']   = tool_input.get('demand_impact')
        row['risk_premium']    = tool_input.get('risk_premium')
        row['magnitude']       = tool_input.get('magnitude')
        row['event_type']      = json.dumps(tool_input.get('event_type') or [])
        row['entities']        = json.dumps(tool_input.get('entities') or [])
        row['certainty']       = tool_input.get('certainty')
        row['time_horizon']    = tool_input.get('time_horizon')
    else:
        for col in ['sentiment_score', 'supply_impact', 'demand_impact',
                    'risk_premium', 'magnitude', 'event_type', 'entities',
                    'certainty', 'time_horizon']:
            row[col] = None

    rows.append(row)

# Save error log regardless of count
ERROR_LOG_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(ERROR_LOG_FILE, 'w') as f:
    json.dump(errors, f, indent=2)

print(f'Parsed:  {len(rows):,} successful results')
print(f'Errors:  {len(errors):,} (saved to {ERROR_LOG_FILE})')
if errors:
    from collections import Counter
    ec = Counter(e['error_type'] for e in errors)
    for etype, cnt in ec.most_common():
        print(f'  {etype}: {cnt}')

Parsed:  107 successful results
Errors:  0 (saved to ../05_reports/calibration/batch_errors.json)


## Cell 9 — Write to llm_features

Writes the parsed rows with `INSERT OR REPLACE` behind a unique index on `article_id`, committing every 500, so the write is idempotent and safe to re-run. Rows already present are skipped.

In [35]:
conn = sqlite3.connect(DB_PATH)

# Ensure unique index so INSERT OR REPLACE is idempotent
conn.execute(
    'CREATE UNIQUE INDEX IF NOT EXISTS idx_llm_article_id ON llm_features(article_id)'
)
conn.commit()

already_written = set(
    r[0] for r in conn.execute(
        'SELECT article_id FROM llm_features WHERE article_id IS NOT NULL'
    ).fetchall()
)

INSERT_SQL = """
    INSERT OR REPLACE INTO llm_features
        (article_id, usable, sentiment_score, supply_impact, demand_impact,
         risk_premium, magnitude, event_type, entities, certainty, time_horizon)
    VALUES
        (:article_id, :usable, :sentiment_score, :supply_impact, :demand_impact,
         :risk_premium, :magnitude, :event_type, :entities, :certainty, :time_horizon)
"""

n_written = n_skipped = 0

for i, row in enumerate(rows):
    if row['article_id'] in already_written:
        n_skipped += 1
        continue
    conn.execute(INSERT_SQL, row)
    n_written += 1
    if n_written % COMMIT_EVERY == 0:
        conn.commit()
        print(f'  [{n_written:,} written | {n_skipped:,} skipped]')

conn.commit()
conn.close()

print(f'\nWrite complete.')
print(f'Written:  {n_written:,}')
print(f'Skipped:  {n_skipped:,} (already present)')


Write complete.
Written:  107
Skipped:  0 (already present)


## Cell 10 — Sanity checks

Reports submission-versus-result totals, `usable` counts, per-channel mean and std on usable articles, the all-channels-zero count (the articles the `usable_strict` filter later drops), the `event_type` and top-entity distributions, and the actual cost recomputed from real token usage.

In [36]:
conn = sqlite3.connect(DB_PATH)

# Submission and result totals
print(f'=== Batch summary ===')
print(f'Articles submitted:   {len(batch_requests):,}')
print(f'Successful results:   {len(rows):,}')
print(f'Errors:               {len(errors):,}')
if errors:
    from collections import Counter
    for etype, cnt in Counter(e["error_type"] for e in errors).most_common():
        print(f'  {etype}: {cnt}')

# usable counts
totals = pd.read_sql("""
    SELECT
        COUNT(*)                                     AS total,
        SUM(CASE WHEN usable=1 THEN 1 ELSE 0 END)   AS usable_true,
        SUM(CASE WHEN usable=0 THEN 1 ELSE 0 END)   AS usable_false
    FROM llm_features
""", conn)
print(f'\n=== llm_features (all rows) ===')
print(totals.to_string(index=False))

# Channel stats on usable articles (pandas — SQLite has no built-in STDEV)
df_usable = pd.read_sql(
    "SELECT sentiment_score, supply_impact, demand_impact, risk_premium "
    "FROM llm_features WHERE usable=1",
    conn,
)
all_zero_count = int(
    ((df_usable['supply_impact'] == 0)
     & (df_usable['demand_impact'] == 0)
     & (df_usable['risk_premium'] == 0)).sum()
)
print(f'\n=== Channel stats (usable=true articles) ===')
for col in ['sentiment_score', 'supply_impact', 'demand_impact', 'risk_premium']:
    print(f'  {col:<18}: mean={df_usable[col].mean():+.4f}  std={df_usable[col].std():.4f}')
print(f'  all_channels_zero : {all_zero_count:,}')

# event_type first-category distribution (usable)
print(f'\n=== event_type (first category) distribution (usable) ===')
ev_rows = conn.execute(
    "SELECT event_type FROM llm_features WHERE usable=1 AND event_type IS NOT NULL"
).fetchall()
from collections import Counter
first_cats = Counter()
for (et,) in ev_rows:
    try:
        cats = json.loads(et)
        if cats:
            first_cats[cats[0]] += 1
    except Exception:
        pass
for cat, cnt in first_cats.most_common():
    print(f'  {cat:<15}: {cnt:,}')

# Top-10 entities
print(f'\n=== Top 10 entities by frequency ===')
ent_rows = conn.execute(
    "SELECT entities FROM llm_features WHERE usable=1 AND entities IS NOT NULL"
).fetchall()
entity_counts = Counter()
for (ent,) in ent_rows:
    try:
        entity_counts.update(json.loads(ent))
    except Exception:
        pass
for ent, cnt in entity_counts.most_common(10):
    print(f'  {ent:<30}: {cnt:,}')

conn.close()

# Actual cost (streams batch results a second time to sum per-result token usage)
PRICE_INPUT_BATCH      = 0.40 / 1e6
PRICE_OUTPUT_BATCH     = 2.00 / 1e6
PRICE_CACHE_WRITE      = 0.50 / 1e6
PRICE_CACHE_READ_BATCH = 0.04 / 1e6

total_input = total_output = total_cache_read = total_cache_write = 0
for result in client.messages.batches.results(batch_id):
    if result.result.type != 'succeeded':
        continue
    usage = result.result.message.usage
    total_input       += usage.input_tokens
    total_output      += usage.output_tokens
    total_cache_read  += getattr(usage, 'cache_read_input_tokens', 0) or 0
    total_cache_write += getattr(usage, 'cache_creation_input_tokens', 0) or 0

actual_cost = (
    total_input        * PRICE_INPUT_BATCH
    + total_output     * PRICE_OUTPUT_BATCH
    + total_cache_read * PRICE_CACHE_READ_BATCH
    + total_cache_write * PRICE_CACHE_WRITE
)
cache_hit_rate = total_cache_read / max(total_cache_read + total_cache_write, 1)

print(f'\n=== Actual cost ===')
print(f'Input tokens:      {total_input:,}')
print(f'Output tokens:     {total_output:,}')
print(f'Cache reads:       {total_cache_read:,}  (hit rate: {cache_hit_rate:.1%})')
print(f'Cache writes:      {total_cache_write:,}')
print(f'ACTUAL TOTAL COST: ${actual_cost:.4f}')

=== Batch summary ===
Articles submitted:   107
Successful results:   107
Errors:               0

=== llm_features (all rows) ===
 total  usable_true  usable_false
 19619        11675          7944

=== Channel stats (usable=true articles) ===
  sentiment_score   : mean=+0.0252  std=0.5014
  supply_impact     : mean=-0.0709  std=0.4629
  demand_impact     : mean=-0.0750  std=0.3396
  risk_premium      : mean=+0.1556  std=0.4257
  all_channels_zero : 1,161

=== event_type (first category) distribution (usable) ===
  geopolitical   : 4,395
  supply         : 3,340
  macro          : 1,689
  demand         : 1,483
  technical      : 413
  other          : 338
  structural     : 16
  political      : 1

=== Top 10 entities by frequency ===
  Iran                          : 3,197
  United States                 : 3,092
  Russia                        : 1,735
  China                         : 1,573
  OPEC+                         : 1,316
  Donald Trump                  : 1,251
  Israel     

## Cell 11 — Logbook update

Appends a dated run summary (batch id, counts, channel stats, actual cost, next steps) to the project logbook.

In [37]:
conn = sqlite3.connect(DB_PATH)
totals_row = conn.execute(
    'SELECT COUNT(*), SUM(usable), SUM(1-usable) FROM llm_features'
).fetchone()
total_lf, n_usable, n_not_usable = totals_row
conn.close()

# Channel stats from df_usable computed in Cell 10 (pandas — SQLite has no STDEV)
s_mean  = round(df_usable['sentiment_score'].mean(), 3)
s_std   = round(df_usable['sentiment_score'].std(), 3)
su_mean = round(df_usable['supply_impact'].mean(), 3)
su_std  = round(df_usable['supply_impact'].std(), 3)
d_mean  = round(df_usable['demand_impact'].mean(), 3)
d_std   = round(df_usable['demand_impact'].std(), 3)
r_mean  = round(df_usable['risk_premium'].mean(), 3)
r_std   = round(df_usable['risk_premium'].std(), 3)

today = datetime.now().strftime('%Y-%m-%d')

entry = f"""
---

## Phase 5 — Full LLM extraction batch ({today})

### Batch details
- Batch ID: `{batch_id}`
- Model: `claude-haiku-4-5`, Batches API (50% discount), prompt caching on system prompt
- SQL filter: `body IS NOT NULL AND body NOT LIKE 'ERROR%'` — no body_valid filter
- Articles submitted: {len(batch_requests):,}
- Successful results: {len(rows):,}
- Errors: {len(errors):,} (logged to `05_reports/calibration/batch_errors.json`)

### Results written to llm_features
- Total rows: {total_lf:,}
- usable=true: {n_usable:,}
- usable=false: {n_not_usable:,}

### Channel stats (usable articles)
- sentiment_score: mean={s_mean}, std={s_std}
- supply_impact:   mean={su_mean}, std={su_std}
- demand_impact:   mean={d_mean}, std={d_std}
- risk_premium:    mean={r_mean}, std={r_std}
- All channels = 0.0: {all_zero_count:,} articles

### Actual cost
- Input tokens: {total_input:,}
- Output tokens: {total_output:,}
- Cache reads: {total_cache_read:,} (hit rate: {cache_hit_rate:.1%})
- Cache writes: {total_cache_write:,}
- **Actual total: ${actual_cost:.4f}**

### Next steps
- Inspect batch_errors.json; resubmit failed articles if meaningful in volume
- Run nb05 (alignment) to refresh liquidity table with new usable-filtered feature coverage
- Re-train TFT on Colab with expanded dataset
"""

with open(LOGBOOK_PATH, 'a') as f:
    f.write(entry)

print(f'Logbook updated: {LOGBOOK_PATH}')

Logbook updated: ../05_reports/thesis/project_logbook.md
